In [3]:
import sys, types, importlib.util
import jax, pandas as pd

# 先放一個空的 qml_benchmarks.model_utils（train 只在 fit() 訓練時才用到）
sys.modules["qml_benchmarks"] = types.ModuleType("qml_benchmarks")
mu = types.ModuleType("qml_benchmarks.model_utils"); mu.train = None
sys.modules["qml_benchmarks.model_utils"] = mu

# 載入下載的 CNN 檔案
spec = importlib.util.spec_from_file_location("cnn", "convolutional_neural_network.py")
cnn = importlib.util.module_from_spec(spec); spec.loader.exec_module(cnn)

# 讀取 8×8 最佳超參數
hp = pd.read_csv("ConvolutionalNeuralNetwork_mnist_pixels_3-5_8x8_GridSearchCV-best-hyperparams.csv")
print(hp)

# 用對方類別自己的初始化（與 fit() 內部相同）；output_channels 不在 grid search 內 → 用類別預設值
model = cnn.ConvolutionalNeuralNetwork(kernel_shape=3)
print("output_channels (default):", model.output_channels)
model.initialize(n_features=64)          # 8×8

for path, leaf in jax.tree_util.tree_flatten_with_path(model.params_)[0]:
    print(jax.tree_util.keystr(path), leaf.shape, leaf.size)
print("Total:", sum(x.size for x in jax.tree_util.tree_leaves(model.params_)))

  hyperparameter  best_value
0     batch_size      32.000
1   kernel_shape       3.000
2  learning_rate       0.001
output_channels (default): [32, 64]
['params']['Conv_0']['bias'] (32,) 32
['params']['Conv_0']['kernel'] (3, 3, 1, 32) 288
['params']['Conv_1']['bias'] (64,) 64
['params']['Conv_1']['kernel'] (3, 3, 32, 64) 18432
['params']['Dense_0']['bias'] (128,) 128
['params']['Dense_0']['kernel'] (256, 128) 32768
['params']['Dense_1']['bias'] (1,) 1
['params']['Dense_1']['kernel'] (128, 1) 128
Total: 51841
